# 01 · Build the raw dataset
Downloads every research-grade iNaturalist photo, from anywhere in the world, of every snake species recorded in India, with a manifest of source, observer, date, place, licence, size and SHA-1 per photo.

**Reads:** iNaturalist API, internet on. **Saves:** `<your username>/serpentguard-indian-snakes`, private. **Time:** about 2.5 hours on CPU.

Species are queried by taxon id, because a name search for *Naja naja* returned a damselfly. Photos from iNaturalist's own server are throttled to 4.5 GB per hour. The team's copy was built on 24 October 2026; a new run downloads more, because iNaturalist keeps growing.

In [ ]:
import collections
import csv
import gzip
import hashlib
import io
import json
import re
import shutil
import threading
import time
import urllib.error
import urllib.request
import zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import kagglehub
from PIL import Image

DATASET_HANDLE = f"{kagglehub.whoami()['username']}/serpentguard-indian-snakes"
SCRATCH = Path("/kaggle/tmp/serpentguard-indian-snakes")
REPORT_DIR = Path("/kaggle/working")
ZIP_NAME = "images.zip"
USER_AGENT = "SerpentGuard-course-project (github.com/Puneet-Bajaj-IITM/Group-5-DS-and-AI-Lab-Project)"
INAT_API = "https://api.inaturalist.org/v1"
INDIA_PLACE_ID = 6681
SERPENTES_TAXON_ID = 85553
SPECIES_PAGE = 500
PER_PAGE = 200
API_PAUSE_S = 1.1
RETRIES = 5
TIMEOUT_S = 90
PHOTO_SIZE = "medium"

S3_HOST = "inaturalist-open-data.s3.amazonaws.com"
FAST_WORKERS = 16
THROTTLED_WORKERS = 4
THROTTLED_MAX_BYTES_PER_HOUR = 4_500_000_000
PROGRESS_EVERY = 5000
FAILURE_SAMPLE = 25
SPECIES_THRESHOLDS = (20, 40, 100, 200, 400)

MANIFEST_FIELDS = (
    "file", "source", "source_record_id", "obs_id", "photo_id", "photo_position",
    "species", "genus", "taxon_id", "record_taxon", "observer_id", "observed_on",
    "created_at", "in_india", "latitude", "longitude", "obscured", "license",
    "attribution", "url", "width", "height", "bytes", "sha1",
)

README = """# SerpentGuard Indian snakes (private, course use only)

Photos of every snake species with research-grade iNaturalist records in India.
Built by the SerpentGuard team (IIT Madras BS, DS and AI Lab, Group 5).

- images.zip: one folder per species, files named <source>_<photo_id>.jpg
- manifest.csv: one row per photo with source, observation, observer, date, place,
  licence, attribution, size and SHA-1
- species.csv: per-species photo counts, total and from India
- build_report.json: counts by source and licence, failures, build time

Source: iNaturalist (research grade, all licences, worldwide photos of Indian species).
Photo copyright stays with each photographer; see the license and attribution columns.
"""


def log(*parts) -> None:
    print(time.strftime("%H:%M:%S"), *parts, flush=True)


def fetch(url: str) -> bytes:
    request = urllib.request.Request(url, headers={"User-Agent": USER_AGENT, "Accept-Encoding": "gzip"})
    for attempt in range(RETRIES):
        try:
            with urllib.request.urlopen(request, timeout=TIMEOUT_S) as response:
                body = response.read()
                if response.headers.get("Content-Encoding") == "gzip":
                    body = gzip.decompress(body)
                return body
        except urllib.error.HTTPError as error:
            if error.code in (403, 404) or attempt == RETRIES - 1:
                raise
        except Exception:
            if attempt == RETRIES - 1:
                raise
        time.sleep(API_PAUSE_S * (attempt + 2))


def fetch_json(url: str) -> dict:
    return json.loads(fetch(url))


def slug(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


class ByteThrottle:
    def __init__(self, max_bytes_per_hour: int):
        self.rate = max_bytes_per_hour / 3600
        self.lock = threading.Lock()
        self.started = time.time()
        self.sent = 0

    def consume(self, n_bytes: int) -> None:
        with self.lock:
            self.sent += n_bytes
            ahead = self.sent / self.rate - (time.time() - self.started)
        if ahead > 0:
            time.sleep(ahead)


class ZipSink:
    def __init__(self, path: Path):
        self.zip = zipfile.ZipFile(path, "w", compression=zipfile.ZIP_STORED, allowZip64=True)
        self.lock = threading.Lock()
        self.names = set()

    def add(self, name: str, body: bytes) -> None:
        with self.lock:
            if name not in self.names:
                self.zip.writestr(name, body)
                self.names.add(name)

    def close(self) -> None:
        self.zip.close()


def indian_species() -> list:
    species, page = [], 1
    while True:
        data = fetch_json(
            f"{INAT_API}/observations/species_counts?place_id={INDIA_PLACE_ID}"
            f"&taxon_id={SERPENTES_TAXON_ID}&quality_grade=research&per_page={SPECIES_PAGE}&page={page}"
        )
        for row in data["results"]:
            taxon = row["taxon"]
            species.append({"name": taxon["name"], "taxon_id": taxon["id"],
                            "common": taxon.get("preferred_common_name"), "india_obs": row["count"]})
        if page * SPECIES_PAGE >= data["total_results"]:
            return species
        page += 1
        time.sleep(API_PAUSE_S)


def inat_records(sp: dict):
    id_above = 0
    while True:
        data = fetch_json(
            f"{INAT_API}/observations?taxon_id={sp['taxon_id']}&quality_grade=research"
            f"&per_page={PER_PAGE}&order_by=id&order=asc&id_above={id_above}"
        )
        time.sleep(API_PAUSE_S)
        for obs in data["results"]:
            user = obs.get("user") or {}
            location = (obs.get("location") or ",").split(",")
            for position, photo in enumerate(obs.get("photos") or []):
                yield {
                    "source": "inaturalist", "source_record_id": obs["id"], "obs_id": obs["id"],
                    "photo_id": photo["id"], "photo_position": position,
                    "species": sp["name"], "taxon_id": sp["taxon_id"], "record_taxon": obs["taxon"]["name"],
                    "observer_id": user.get("id"), "observed_on": obs.get("observed_on"),
                    "created_at": obs.get("created_at"),
                    "in_india": INDIA_PLACE_ID in (obs.get("place_ids") or []),
                    "latitude": location[0], "longitude": location[1], "obscured": obs.get("obscured"),
                    "license": photo.get("license_code") or "all-rights-reserved",
                    "attribution": photo.get("attribution"),
                    "url": photo["url"].replace("/square.", f"/{PHOTO_SIZE}."),
                }
        if len(data["results"]) < PER_PAGE:
            return
        id_above = data["results"][-1]["id"]


def is_unthrottled(record: dict) -> bool:
    return S3_HOST in (record["url"] or "")


def store(record: dict, sink: ZipSink, throttle: ByteThrottle | None) -> dict:
    entry = f"{slug(record['species'])}/{record['source']}_{record['photo_id']}.jpg"
    name = f"images/{entry}"
    try:
        body = fetch(record["url"] or "")
        if throttle:
            throttle.consume(len(body))
        with Image.open(io.BytesIO(body)) as image:
            width, height = image.size
            image.verify()
        sink.add(entry, body)
        return dict(record, file=name, width=width, height=height, bytes=len(body),
                    sha1=hashlib.sha1(body).hexdigest(), genus=record["species"].split()[0], ok=True)
    except Exception as error:
        return dict(record, ok=False, error=repr(error)[:200])


def run_pool(batch: list, workers: int, sink: ZipSink, throttle, label: str, out: list, lock: threading.Lock) -> None:
    log(f"{label}: {len(batch)} photos")
    with ThreadPoolExecutor(workers) as pool:
        for i, result in enumerate(pool.map(lambda r: store(r, sink, throttle), batch), 1):
            with lock:
                out.append(result)
            if i % PROGRESS_EVERY == 0:
                log(f"{label}: {i}/{len(batch)}")
    log(f"{label}: done")


def download_all(records: list, sink: ZipSink) -> list:
    fast = [r for r in records if is_unthrottled(r)]
    slow = [r for r in records if not is_unthrottled(r)]
    results, lock = [], threading.Lock()
    jobs = (
        threading.Thread(target=run_pool, args=(fast, FAST_WORKERS, sink, None, "fast", results, lock)),
        threading.Thread(target=run_pool, args=(slow, THROTTLED_WORKERS, sink,
                                                ByteThrottle(THROTTLED_MAX_BYTES_PER_HOUR), "throttled", results, lock)),
    )
    for job in jobs:
        job.start()
    for job in jobs:
        job.join()
    return results


def write_outputs(work_dir: Path, species: list, results: list, started: float) -> dict:
    ok = [r for r in results if r["ok"]]
    with open(work_dir / "manifest.csv", "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=MANIFEST_FIELDS, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(ok)
    per_species = collections.Counter(r["species"] for r in ok)
    per_species_india = collections.Counter(r["species"] for r in ok if r["in_india"])
    with open(work_dir / "species.csv", "w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(["species", "genus", "taxon_id", "common_name", "inat_india_obs", "photos", "photos_in_india"])
        for sp in sorted(species, key=lambda s: -per_species[s["name"]]):
            writer.writerow([sp["name"], sp["name"].split()[0], sp["taxon_id"], sp["common"],
                             sp["india_obs"], per_species[sp["name"]], per_species_india[sp["name"]]])
    report = {
        "built_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "minutes": round((time.time() - started) / 60, 1),
        "species_in_india": len(species),
        "species_with_photos": len(per_species),
        "photos_ok": len(ok),
        "photos_failed": len(results) - len(ok),
        "bytes_total": sum(r["bytes"] for r in ok),
        "by_source": collections.Counter(r["source"] for r in ok).most_common(),
        "by_license": collections.Counter(r["license"] for r in ok).most_common(),
        "species_with_at_least": {t: sum(c >= t for c in per_species.values()) for t in SPECIES_THRESHOLDS},
        "duplicate_sha1": len(ok) - len({r["sha1"] for r in ok}),
        "failures_sample": [r for r in results if not r["ok"]][:FAILURE_SAMPLE],
    }
    (work_dir / "build_report.json").write_text(json.dumps(report, indent=1, default=str))
    (work_dir / "README.md").write_text(README)
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    for name in ("build_report.json", "species.csv", "manifest.csv"):
        shutil.copy(work_dir / name, REPORT_DIR / name)
    return report


def upload(work_dir: Path) -> None:
    kagglehub.dataset_upload(DATASET_HANDLE, str(work_dir),
                             version_notes=f"build {time.strftime('%Y-%m-%d %H:%M UTC', time.gmtime())}")


def main() -> None:
    started = time.time()
    work_dir = SCRATCH
    work_dir.mkdir(parents=True, exist_ok=True)
    log(f"work dir {work_dir}")

    species = indian_species()
    log(f"{len(species)} Indian species")

    records = []
    for i, sp in enumerate(species, 1):
        records.extend(inat_records(sp))
        if i % 25 == 0:
            log(f"manifest {i}/{len(species)} species, {len(records)} photo records")
    log(f"{len(records)} photo records: {collections.Counter(r['source'] for r in records).most_common()}")

    sink = ZipSink(work_dir / ZIP_NAME)
    try:
        results = download_all(records, sink)
    finally:
        sink.close()
    report = write_outputs(work_dir, species, results, started)
    log("report:", json.dumps({k: v for k, v in report.items() if k != "failures_sample"}, default=str))
    upload(work_dir)
    log(f"uploaded {DATASET_HANDLE}")

In [ ]:
main()